In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [3]:
train_df = pd.read_csv("data/train_v2.csv")
train_df.head()

,msno,is_churn
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,1


In [8]:
print(f"The shape of train_df: {train_df.shape}")
train_df.isna().sum()

The shape of train_df: (970960, 2)


msno        0
is_churn    0
dtype: int64

In [12]:
print("----- Churn Ratio -----")
print(train_df["is_churn"].value_counts(normalize=True) * 100)

----- Churn Ratio -----
is_churn
0    91.005809
1     8.994191
Name: proportion, dtype: float64


In [15]:
# peek into users_log without memory load
logs_peek = pd.read_csv("data/user_logs_v2.csv", nrows=5)
print("\n--- User Logs Peek ---")
print(logs_peek)
print("\n--- Logs Columns & Types ---")
print(logs_peek.dtypes)


--- User Logs Peek ---
                                           msno      date  num_25  num_50  \
0  u9E91QDTvHLq6NXjEaWv8u4QIqhrHk72kE+w31Gnhdg=  20170331       8       4   
1  nTeWW/eOZA/UHKdD5L7DEqKKFTjaAj3ALLPoAWsU8n0=  20170330       2       2   
2  2UqkWXwZbIjs03dHLU9KHJNNEvEkZVzm69f3jCS+uLI=  20170331      52       3   
3  ycwLc+m2O0a85jSLALtr941AaZt9ai8Qwlg9n0Nql5U=  20170331     176       4   
4  EGcbTofOSOkMmQyN1NMLxHEXJ1yV3t/JdhGwQ9wXjnI=  20170331       2       1   

   num_75  num_985  num_100  num_unq  total_secs  
0       0        1       21       18    6309.273  
1       1        0        9       11    2390.699  
2       5        3       84      110   23203.337  
3       2        2       19      191    7100.454  
4       0        1      112       93   28401.558  

--- Logs Columns & Types ---
msno              str
date            int64
num_25          int64
num_50          int64
num_75          int64
num_985         int64
num_100         int64
num_unq         int64
t

In [20]:
logs_peek.shape

(5, 9)

In [18]:
# Now we are splitting data with sklearn

sample_df, _ = train_test_split(
    train_df,
    train_size=50000,
    stratify=train_df["is_churn"],
    random_state=42
)

In [19]:
print("Original Ratio")
print(train_df['is_churn'].value_counts(normalize=True))

print("\nSample Ratio")
print(sample_df['is_churn'].value_counts(normalize=True))

Original Ratio
is_churn
0    0.910058
1    0.089942
Name: proportion, dtype: float64

Sample Ratio
is_churn
0    0.91006
1    0.08994
Name: proportion, dtype: float64


In [26]:
import os
from tqdm import tqdm

# 1. Pipeline Paths & Configurations
input_path = 'data/user_logs_v2.csv'  # Path verify kar lena agar file seedha data/ mein ho
output_dir = 'data/processed'
output_path = os.path.join(output_dir, 'sampled_logs.csv')

# Ensure output directory exists to avoid FileNotFoundError
os.makedirs(output_dir, exist_ok=True)

# 2. Extract Target Cohort as Hash Set (O(1) lookups)
target_set = set(sample_df['msno'])
print(f"Target cohort size: {len(target_set):,} unique users")

# 3. Stream & Filter Parameters
chunk_size = 500_000
total_estimated_rows = 18_396_362
total_chunks = (total_estimated_rows // chunk_size) + 1

# Reset destination file if already exists
if os.path.exists(output_path):
    os.remove(output_path)

first_write = True
total_matched_rows = 0

# 4. Ingestion & Filtering Loop
with tqdm(total=total_chunks, desc="Streaming & Filtering logs") as pbar:
    for chunk in pd.read_csv(input_path, chunksize=chunk_size):
        # Fast membership check
        filtered_chunk = chunk[chunk['msno'].isin(target_set)]
        
        if not filtered_chunk.empty:
            total_matched_rows += len(filtered_chunk)
            filtered_chunk.to_csv(
                output_path,
                mode='a',
                header=first_write,
                index=False
            )
            first_write = False
            
        pbar.update(1)

print(f"\nPipeline Completed Successfully!")
print(f"Total Rows Extracted: {total_matched_rows:,}")
print(f"Output File Size: {os.path.getsize(output_path) / (1024 * 1024):.2f} MB")

Target cohort size: 50,000 unique users


Streaming & Filtering logs: 100%|██████████| 37/37 [01:47<00:00,  2.91s/it]


Pipeline Completed Successfully!
Total Rows Extracted: 695,320
Output File Size: 51.45 MB


In [46]:
sampled_logs = pd.read_csv("data/processed/sampled_logs.csv")
sampled_logs["msno"].nunique()

38750

In [47]:
sampled_logs.head()

,msno,date,num_25,num_50,num_75,num_985,num_100,num_unq,total_secs
0,b74Xot5HcOL8YQGiuwLw+Cm7RyGV2fOkGczJomGLocg=,20170318,25,8,6,3,210,165,54995.604
1,qJ5QfEWdjA25q58/I1PxXswjWGkta6lTpsMfzvx7bQs=,20170310,12,2,2,0,44,54,11145.656
2,bLf43Q9GcbNT+8UVbhTZ41lXUnoXsHQDIVSuWVviYhg=,20170329,0,0,1,0,33,30,8457.177
3,bB/I5ukaNnz6ZCxMQj1Zg2EUF9jfJyaRKECS/bTfssA=,20170304,173,24,7,1,2,186,5506.589
4,6jNgQq6/hnIKE/lVz1eV/lsnQyElJw0+FC32/0g5v6c=,20170304,3,0,1,0,64,52,14907.321


In [48]:
# 1. Feature Aggregation (1-Row-Per-User)
user_features = sampled_logs.groupby("msno").agg(
    active_days=("date", "count"),
    last_active_raw=("date", "max"),
    total_secs_sum=("total_secs", "sum"),
    total_secs_mean=("total_secs", "mean"),
    total_unique_songs=("num_unq", "sum"),
    num_25_sum=("num_25", "sum"),
    num_100_sum=("num_100", "sum")
).reset_index()

user_features.head()

,msno,active_days,last_active_raw,total_secs_sum,total_secs_mean,total_unique_songs,num_25_sum,num_100_sum
0,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,28,20170331,115411.260,4121.830714,468,43,485
1,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,21,20170330,149896.558,7137.931333,828,207,436
2,++HYwmBh/zxrF/wAgCF/TCntq5vjc6TpCEFWTz6AzaI=,26,20170331,336481.480,12941.595385,1434,306,1271
3,++l8WoNUmsqs7C9ZVyk3pdxXklhdJcSrj50v5su2x/w=,7,20170328,16833.367,2404.766714,74,12,67
4,++lyY7knLqnnBcAjwVDpc3hPv5RGCuagXv7H50Y2e4I=,28,20170331,158906.020,5675.215000,716,205,495


In [49]:
# 2. Behavioral Ratios & Recency Math
# Snapshot date KKBox logs ki March 31, 2017 hai
user_features['last_active_date'] = pd.to_datetime(user_features['last_active_raw'].astype(str), format='%Y%m%d')

In [50]:
ref_date = pd.to_datetime('20170331', format='%Y%m%d')
user_features['days_since_last_active'] = (ref_date - user_features['last_active_date']).dt.days

In [51]:
# Temporary date columns drop kar diye (model sirf numbers consume karega)
user_features.drop(columns=['last_active_raw', 'last_active_date'], inplace=True)

In [52]:
# Frustration vs Completion Ratios
total_songs = user_features['num_25_sum'] + user_features['num_100_sum']
user_features['skip_ratio'] = np.where(total_songs > 0, user_features['num_25_sum'] / total_songs, 0.0)
user_features['completion_ratio'] = np.where(total_songs > 0, user_features['num_100_sum'] / total_songs, 0.0)

In [53]:
# 3. Ground Truth Merge (Left Join)
print("Merging with 50,000 cohort ground truth...")
feature_matrix = sample_df.merge(user_features, on='msno', how='left')

print(f"Final Dataset Shape: {final_df.shape}")
print(f"Total Unique Users: {final_df['msno'].nunique():,}")

Merging with 50,000 cohort ground truth...
Final Dataset Shape: (50000, 11)
Total Unique Users: 50,000


In [54]:
# 4. MISSING DATA HANDLING (The "Belt & Suspenders" Pattern)
# A. Explicit Binary Indicator: 1 for Ghost Users (0 logs), 0 for Active Users
feature_matrix['is_ghost'] = feature_matrix['active_days'].isna().astype(int)

# B. Recency Penalty Imputation (Window limit + 1 day = 31 days)
feature_matrix['days_since_last_active'] = feature_matrix['days_since_last_active'].fillna(31.0)

# C. Zero-Imputation for engagement volumes, durations, and ratios
zero_impute_cols = [
    'active_days',
    'total_secs_sum',
    'total_secs_mean',
    'total_unique_songs',
    'num_25_sum',
    'num_100_sum',
    'skip_ratio',
    'completion_ratio'
]
feature_matrix[zero_impute_cols] = feature_matrix[zero_impute_cols].fillna(0.0)

# 5. SANITY AUDIT & VERIFICATION
print("\nFeature Matrix Built Successfully!")
print(f"Shape: {feature_matrix.shape} (Expected: 50,000 rows)")
print(f"Total Unique Users: {feature_matrix['msno'].nunique():,}")
print(f"Ghost Users Detected: {feature_matrix['is_ghost'].sum():,} ({feature_matrix['is_ghost'].mean()*100:.1f}%)")
print(f"Remaining Nulls in DataFrame: {feature_matrix.isna().sum().sum()}")


Feature Matrix Built Successfully!
Shape: (50000, 12) (Expected: 50,000 rows)
Total Unique Users: 50,000
Ghost Users Detected: 11,250 (22.5%)
Remaining Nulls in DataFrame: 0


In [59]:
feature_matrix.head()

,msno,is_churn,active_days,total_secs_sum,total_secs_mean,total_unique_songs,num_25_sum,num_100_sum,days_since_last_active,skip_ratio,completion_ratio,is_ghost
0,/ztSaa8pxgwR1gn0nPDOEyasB5MyAOOvNN7RC8Swzog=,0,20.0,118249.753,5912.487650,604.0,169.0,425.0,0.0,0.284512,0.715488,0
1,b5IOGXSw8bf6B4OhXK53m7BJNVWRXELR1J/NWddi150=,0,20.0,93642.378,4682.118900,398.0,114.0,361.0,0.0,0.240000,0.760000,0
2,7wQNGhJPkJOrdBoT2uVYvtDlPEfsZLpql1oUkQHf9sU=,0,26.0,565734.520,21759.020000,1418.0,41.0,2369.0,0.0,0.017012,0.982988,0
3,IAEc2jQfBCW/kzxjwSw7ia6jSfe3pY73btVZHtFCz6U=,0,15.0,86380.246,5758.683067,374.0,118.0,319.0,2.0,0.270023,0.729977,0
4,TbkN2Bk9uDUDfz76t8PeSaBbAmlMhKwY4WInrzJ0+AA=,0,0.0,0.000,0.000000,0.0,0.0,0.0,31.0,0.000000,0.000000,1


In [61]:
sample_df.shape

(50000, 2)

In [62]:
feature_matrix.shape

(50000, 12)